# Q1 - What is this dataset about? What is the scope?

Goal: describe what one row means, which entities and time span are covered, and how far the sample can be generalised.

Conventions used in this notebook:
- `polars` only, no pandas.
- Every helper is a typed, documented, side-effect-free function: DataFrame in, DataFrame out.
- Constants are `UPPER_SNAKE_CASE`; DataFrames are prefixed `df_` and named after their grain (`df_player_matches`, `df_players`, `df_matches`).

## 0. Setup

In [ ]:
from pathlib import Path

import polars as pl

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(60)

DATA_PATH: Path = Path('playersmatches_test - playersmatches_test.csv')
ID_SEPARATOR: str = ','
NULL_LITERALS: list[str] = ['null', 'NULL', 'None', 'NaN', '']
INTEGER_COLUMNS: list[str] = ['kills', 'death', 'playernb_in_match']

# (inclusive upper bound, label), evaluated in order; anything above falls into OVERFLOW_BUCKET_LABEL
MATCH_COUNT_BUCKETS: list[tuple[int, str]] = [
    (1, '1'),
    (2, '2'),
    (4, '3-4'),
    (9, '5-9'),
    (19, '10-19'),
    (49, '20-49'),
]
OVERFLOW_BUCKET_LABEL: str = '50+'

## 1. Helpers

In [ ]:
def parse_bool(column: str) -> pl.Expr:
    """Build an expression that converts a TRUE/FALSE string column to Boolean.

    Args:
        column: Name of a String column holding `TRUE` / `FALSE`.

    Returns:
        A Boolean expression named `column`; values other than TRUE/FALSE become null.
    """
    upper: pl.Expr = pl.col(column).str.to_uppercase()
    return (
        pl.when(upper == 'TRUE').then(True)
        .when(upper == 'FALSE').then(False)
        .otherwise(None)
        .alias(column)
    )


def load_player_matches(path: Path) -> pl.DataFrame:
    """Load the raw player-match CSV and normalise types.

    `victory` is not boolean in the source: it also contains `draw`, so it is
    exposed as a three-state `outcome` (win / loss / draw).

    Args:
        path: Location of the CSV file.

    Returns:
        One row per (player, match) with `match_datetime` as a UTC datetime,
        `outcome` as String and `still_active` as Boolean. The raw `victory`
        column is dropped.
    """
    df_raw: pl.DataFrame = pl.read_csv(
        path,
        infer_schema=False,  # source mixes literals ('null', 'draw') into typed columns
        null_values=NULL_LITERALS,
    )
    victory: pl.Expr = pl.col('victory').str.to_lowercase()
    return df_raw.with_columns(
        pl.col('match_datetime').str.to_datetime(time_zone='UTC', strict=False),
        pl.when(victory == 'true').then(pl.lit('win'))
        .when(victory == 'false').then(pl.lit('loss'))
        .when(victory == 'draw').then(pl.lit('draw'))
        .otherwise(pl.lit(None, dtype=pl.String))
        .alias('outcome'),
        parse_bool('still_active'),
        pl.col(INTEGER_COLUMNS).cast(pl.Int64, strict=False),
    ).drop('victory')


def count_ids(column: str, separator: str = ID_SEPARATOR) -> pl.Expr:
    """Build an expression counting the ids stored in a delimited string column.

    Args:
        column: Name of the column holding separator-joined ids.
        separator: Delimiter between ids.

    Returns:
        An integer expression; null or empty strings count as 0.
    """
    ids: pl.Expr = pl.col(column).fill_null('')
    return pl.when(ids == '').then(0).otherwise(ids.str.count_matches(separator) + 1)


def bucket_count(
    column: str,
    buckets: list[tuple[int, str]],
    overflow_label: str,
) -> pl.Expr:
    """Build an expression mapping an integer column to ordered label buckets.

    Args:
        column: Name of the integer column to bucket.
        buckets: Ordered (inclusive upper bound, label) pairs.
        overflow_label: Label for values above every upper bound.

    Returns:
        A string expression holding the bucket label.
    """
    expression: pl.Expr | None = None
    for upper_bound, label in buckets:
        condition: pl.Expr = pl.col(column) <= upper_bound
        if expression is None:
            expression = pl.when(condition).then(pl.lit(label))
        else:
            expression = expression.when(condition).then(pl.lit(label))
    return expression.otherwise(pl.lit(overflow_label))


def profile_schema(df: pl.DataFrame) -> pl.DataFrame:
    """Summarise dtype, null count and cardinality for every column.

    Args:
        df: Any DataFrame.

    Returns:
        One row per column of `df`.
    """
    return pl.DataFrame(
        {
            'column': df.columns,
            'dtype': [str(dtype) for dtype in df.dtypes],
            'null_count': [df[name].null_count() for name in df.columns],
            'null_pct': [round(df[name].null_count() / df.height * 100, 2) for name in df.columns],
            'n_unique': [df[name].n_unique() for name in df.columns],
        }
    )


def share_by_category(df: pl.DataFrame, column: str) -> pl.DataFrame:
    """Count rows per distinct value of `column` with their percentage share.

    Args:
        df: Input DataFrame.
        column: Categorical column to summarise.

    Returns:
        Columns `column`, `row_count`, `share_pct`, sorted by `row_count` descending.
    """
    return (
        df.group_by(column)
        .agg(pl.len().alias('row_count'))
        .with_columns((pl.col('row_count') / df.height * 100).round(1).alias('share_pct'))
        .sort('row_count', descending=True)
    )


def build_monthly_volume(df_player_matches: pl.DataFrame) -> pl.DataFrame:
    """Aggregate rows, distinct players and distinct matches per calendar month.

    Args:
        df_player_matches: One row per (player, match).

    Returns:
        One row per `YYYY-MM`, sorted chronologically.
    """
    return (
        df_player_matches.group_by(pl.col('match_datetime').dt.strftime('%Y-%m').alias('month'))
        .agg(
            pl.len().alias('row_count'),
            pl.col('playerid').n_unique().alias('player_count'),
            pl.col('matchid').n_unique().alias('match_count'),
        )
        .sort('month')
    )


def build_players(df_player_matches: pl.DataFrame) -> pl.DataFrame:
    """Collapse the match log to one row per player.

    Args:
        df_player_matches: One row per (player, match).

    Returns:
        One row per player with match count, first/last match and the
        `still_active` flag (plus its distinct-value count to detect inconsistency).
    """
    return df_player_matches.group_by('playerid').agg(
        pl.len().alias('match_count'),
        pl.col('match_datetime').min().alias('first_match_at'),
        pl.col('match_datetime').max().alias('last_match_at'),
        pl.col('still_active').n_unique().alias('still_active_distinct_count'),
        pl.col('still_active').first().alias('still_active'),
    )


def build_match_count_distribution(df_players: pl.DataFrame) -> pl.DataFrame:
    """Distribute players over match-count buckets.

    Args:
        df_players: One row per player with a `match_count` column.

    Returns:
        One row per bucket, ordered from fewest to most matches.
    """
    return (
        df_players.with_columns(
            bucket_count('match_count', MATCH_COUNT_BUCKETS, OVERFLOW_BUCKET_LABEL).alias('match_count_bucket')
        )
        .group_by('match_count_bucket')
        .agg(
            pl.len().alias('player_count'),
            pl.col('match_count').min().alias('_sort_key'),
        )
        .with_columns((pl.col('player_count') / df_players.height * 100).round(1).alias('share_pct'))
        .sort('_sort_key')
        .drop('_sort_key')
    )


def add_roster_columns(df_player_matches: pl.DataFrame) -> pl.DataFrame:
    """Add teammate/opponent counts and a roster consistency flag.

    Args:
        df_player_matches: One row per (player, match).

    Returns:
        Input plus `teammate_count`, `opponent_count` and `is_roster_consistent`
        (1 + teammates + opponents == `playernb_in_match`).
    """
    return df_player_matches.with_columns(
        count_ids('teammateid').alias('teammate_count'),
        count_ids('opponentid').alias('opponent_count'),
    ).with_columns(
        (
            (1 + pl.col('teammate_count') + pl.col('opponent_count')) == pl.col('playernb_in_match')
        ).alias('is_roster_consistent')
    )


def build_match_coverage(df_player_matches: pl.DataFrame) -> pl.DataFrame:
    """Compare observed participants to announced participants per match.

    Args:
        df_player_matches: One row per (player, match).

    Returns:
        One row per match with `observed_count`, `announced_count` and
        `coverage_ratio` (observed / announced).
    """
    return df_player_matches.group_by('matchid').agg(
        pl.len().alias('observed_count'),
        pl.col('playernb_in_match').max().alias('announced_count'),
    ).with_columns((pl.col('observed_count') / pl.col('announced_count')).alias('coverage_ratio'))

## 2. Load

In [ ]:
df_player_matches: pl.DataFrame = load_player_matches(DATA_PATH)
print(df_player_matches.shape)
df_player_matches.head(3)

## 3. Structure: columns and types

In [ ]:
profile_schema(df_player_matches)

Data dictionary (as inferred, to confirm with the data owner):

| column | meaning |
|---|---|
| playerid | anonymised player id |
| matchid | anonymised match id |
| gamemode | PVP / PVP_RANKED / PVP_UNRANKED |
| victory | match result: TRUE / FALSE / draw (exposed as `outcome`: win / loss / draw) |
| kills, death | player's kills and deaths in the match |
| match_datetime | match timestamp (UTC) |
| playernb_in_match | number of players in the match |
| still_active | player-level flag, likely the churn label (TRUE = still playing) |
| teammateid, opponentid | comma-separated player ids |

## 4. Grain: what does one row represent?

In [ ]:
unique_pair_count: int = df_player_matches.select(['playerid', 'matchid']).unique().height
print(f'rows                   : {df_player_matches.height}')
print(f'unique (player, match) : {unique_pair_count}')
print(f'duplicated pairs       : {df_player_matches.height - unique_pair_count}')
print(f'unique players         : {df_player_matches["playerid"].n_unique()}')
print(f'unique matches         : {df_player_matches["matchid"].n_unique()}')

If duplicated pairs = 0, the grain is **one row per player per match**.

## 5. Scope: time, game modes, volume

In [ ]:
first_match_at = df_player_matches['match_datetime'].min()
last_match_at = df_player_matches['match_datetime'].max()
print(f'first match       : {first_match_at}')
print(f'last match        : {last_match_at}')
print(f'span (days)       : {(last_match_at - first_match_at).days}')
print(f'unparseable dates : {df_player_matches["match_datetime"].null_count()}')

build_monthly_volume(df_player_matches)

In [ ]:
share_by_category(df_player_matches, 'gamemode')

In [ ]:
print(share_by_category(df_player_matches, 'outcome'))
print(share_by_category(df_player_matches, 'playernb_in_match').sort('playernb_in_match'))

## 6. Scope: how much history per player?

In [ ]:
df_players: pl.DataFrame = build_players(df_player_matches)

print(df_players['match_count'].describe(percentiles=[0.5, 0.75, 0.9, 0.99]))
build_match_count_distribution(df_players)

In [ ]:
inconsistent_player_count: int = df_players.filter(pl.col('still_active_distinct_count') > 1).height
print(f'players with inconsistent still_active: {inconsistent_player_count}')

print(share_by_category(df_players, 'still_active'))

df_players.group_by('still_active').agg(
    pl.col('last_match_at').min().alias('last_match_min'),
    pl.col('last_match_at').median().alias('last_match_median'),
    pl.col('last_match_at').max().alias('last_match_max'),
)

The last table checks how `still_active` relates to the last match date. If active players' last match is much later than churned players', the flag is likely derived from recency, which has leakage implications for Q6.

## 7. Scope: is each match fully observed?

Compares the number of participants present in the file with the number announced in `playernb_in_match`. A low ratio means the file is a player sample, not a full match log.

In [ ]:
df_player_matches = add_roster_columns(df_player_matches)

roster_consistency_rate: float = df_player_matches['is_roster_consistent'].mean()
print(f'roster arithmetic holds: {roster_consistency_rate:.4f}')

(
    df_player_matches.group_by(['teammate_count', 'opponent_count'])
    .agg(pl.len().alias('row_count'))
    .sort('row_count', descending=True)
)

In [ ]:
df_matches: pl.DataFrame = build_match_coverage(df_player_matches)

print(df_matches['coverage_ratio'].describe())
fully_observed_rate: float = (df_matches['coverage_ratio'] >= 1).mean()
print(f'share of matches fully observed: {fully_observed_rate:.3f}')

In [ ]:
# How many co-players referenced in the id lists are themselves players in the file?
known_player_ids: pl.Series = df_player_matches['playerid'].unique()
referenced_player_ids: pl.Series = (
    pl.concat([df_player_matches['teammateid'], df_player_matches['opponentid']])
    .drop_nulls()
    .str.split(ID_SEPARATOR)
    .explode()
    .unique()
)
print(f'distinct co-players referenced  : {referenced_player_ids.len()}')
print(f'of which present as a row owner : {referenced_player_ids.is_in(known_player_ids.implode()).sum()}')

## 8. Answer

**What it is:** a match-participation log of a team-based PvP shooter (mostly 5v5). One row per player per match, with outcome, kills/deaths, game mode, timestamp, roster ids (teammates / opponents) and a player-level `still_active` flag, which is most likely the churn label.

**Scope**
- **Period:** 2025-03-04 to 2026-03-03 (about 12 months; March 2026 is a partial month).
- **Volume:** 95,733 rows, 840 distinct players, 95,722 distinct matches.
- **Modes:** PVP_RANKED 50.7%, PVP 33.7%, PVP_UNRANKED 15.6%, OPERATIONS 0.005% (5 rows).
- **Outcome:** 52.0% loss, 48.0% win, 43 draws, 13 missing.
- **History depth per player:** median 28 matches, p90 337, max 1,974. 11% of players have a single match; 40% have 50+.
- **Label:** `still_active` is constant per player (0 inconsistencies). 41.9% TRUE (352 players), 58.1% FALSE (488 players).

**Key scope finding: this is a player-based sample, not a match log.**
- Roughly 10 players per match are announced, but on average only about 1 appears in the file (mean coverage 9.7%).
- Of 710,274 distinct co-players referenced in teammate/opponent lists, only 21 are among the 840 players in the file.
- So the file holds the full history of 840 sampled players, with their match partners listed as ids but not present as rows.

**Caveats for later questions**
- Teammate / opponent graph features cannot be built from joins. Only per-player aggregates over the id lists are possible (unique co-players, repeat-teammate rate).
- `still_active` is a snapshot. Active players' median last match is 2026-02-14 vs 2025-09-13 for churned players, so the flag is clearly recency-driven. Recency features leak the label (see Q6).
- Players with 1-4 matches are 24% of the sample, so per-player ratios (win rate, K/D) are noisy for them.
- `victory` is not boolean (`draw`), and `kills`/`death` have 153 nulls, which Q2 must handle.
